# BABILong 64k C1/C2 construction robustness
Full frozen 32-example evicted cohort.

## Setup: imports, cohort, model, J-Lens


In [1]:
from pathlib import Path
import sys, json, time, math, statistics as st, torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download

REPO = Path.cwd()
if not (REPO / "ahn_interp.py").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import ahn_interp as ai
import ruler_controls as rc

METHOD = "babilong_c1c2_pointcapture_v3"
RUN = "run_3b_gdn"
SCREEN_WINDOW = 32640

OUT = REPO / "results/babilong/07c_babilong_c1c2_full.json"
COHORT = REPO / "results/babilong/07_babilong_64k_evicted_cohort.json"
LENS = REPO / "results/run_3b_gdn/jlens_qwen25_3b_1000ctx.pt"
VALIDATION = REPO / "results/run_3b_gdn/02_table3_jlens_validation_1000ctx.json"

def loadj(path):
    with open(path) as f:
        return json.load(f)

def savej(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    tmp.replace(path)

# Cohort
cohort = loadj(COHORT)
IDS = [int(x) for x in cohort["candidate_ids"]]
META = {int(r["id"]): r for r in cohort["rows"]}

assert len(IDS) == 32
assert all(
    int(META[i]["support_distance_from_end"]) > SCREEN_WINDOW
    for i in IDS
)

# Run config
CFG = ai.load_run_config(RUN)
LAYERS = list(CFG["layers"])
SINKS = int(CFG["num_attn_sinks"])
WINDOW = int(CFG["sliding_window"])

assert WINDOW <= SCREEN_WINDOW, (
    f"Model window {WINDOW} > screening window {SCREEN_WINDOW}"
)

bad_sink = []
for i in IDS:
    pos = (
        int(META[i]["context_tokens"])
        - int(META[i]["support_distance_from_end"])
    )
    if pos < SINKS:
        bad_sink.append((i, pos))

assert not bad_sink, f"sink-region supports found: {bad_sink}"

# BABILong
data_path = hf_hub_download(
    repo_id="RMT-team/babilong",
    filename="data/qa1/64k.json",
    repo_type="dataset",
)
ds = load_dataset("json", data_files={"qa1": data_path})["qa1"]

# Model
bundle = ai.load_ahn_model(
    CFG["model_path"],
    dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=WINDOW,
    num_attn_sinks=SINKS,
)

tok, model = bundle.tokenizer, bundle.model
DEVICE = next(model.parameters()).device

# J-Lens
lens = ai.JacobianLens.load(
    ai.resolve_lens_path(str(LENS)),
    map_location=str(DEVICE),
)

assert set(LAYERS) <= set(lens.jacobians)
assert all(hasattr(model.model.layers[L], "ahn") for L in LAYERS)

LENS_VALIDATED = bool(loadj(VALIDATION).get("TABLE_3_PASSED", False))
CHANCE = rc.CHANCE
C2_BAR = rc.C2_BAR

# Targets
TARGETS = sorted({ds[i]["target"].strip() for i in IDS})

def target_id(target):
    ids = tok.encode(" " + target, add_special_tokens=False)
    q = tok.encode("?", add_special_tokens=False)
    qa = tok.encode("? " + target, add_special_tokens=False)

    assert len(ids) == 1 and qa == q + ids, (target, ids, qa)
    return int(ids[0])

TARGET_IDS = {t: target_id(t) for t in TARGETS}

ANS = {
    i: {
        "target": ds[i]["target"].strip(),
        "id": TARGET_IDS[ds[i]["target"].strip()],
    }
    for i in IDS
}

CAND_IDS = sorted(set(TARGET_IDS.values()))
CAND_T = torch.tensor(CAND_IDS, device=DEVICE, dtype=torch.long)

/home/jupyter-dphs-29af/ahn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.55it/s]


## Hooks + input construction


In [2]:
@torch.inference_mode()
def capture_point(inputs, pos, nowrite):
    captured, handles = {}, []

    def zero_hook(module, inp, out):
        if isinstance(out, tuple):
            return (torch.zeros_like(out[0]),) + out[1:]
        return torch.zeros_like(out)

    def point_hook(L):
        def hook(module, inp, out):
            captured[L] = inp[0][0, pos].detach().float().cpu().clone()
        return hook

    for L in LAYERS:
        layer = model.model.layers[L]

        if nowrite:
            handles.append(layer.ahn.register_forward_hook(zero_hook))

        handles.append(
            layer.post_attention_layernorm.register_forward_hook(point_hook(L))
        )

    try:
        model(**inputs, use_cache=True, num_logits_to_keep=1)
    finally:
        for h in handles:
            h.remove()

    if set(captured) != set(LAYERS):
        raise RuntimeError(
            f"captured={sorted(captured)}, expected={LAYERS}"
        )

    return captured


def make_input(idx):
    ex = ds[idx]
    text = ex["input"].rstrip() + "\n" + ex["question"].strip()

    enc = tok(text, return_tensors="pt")
    P = enc["input_ids"].shape[1]
    tid = ANS[idx]["id"]

    target = torch.tensor([[tid]], dtype=enc["input_ids"].dtype)

    input_ids = torch.cat([enc["input_ids"], target], dim=1)

    attention_mask = torch.cat([
        enc["attention_mask"],
        torch.ones((1, 1), dtype=enc["attention_mask"].dtype),
    ], dim=1)

    inputs = {
        "input_ids": input_ids.to(DEVICE),
        "attention_mask": attention_mask.to(DEVICE),
    }

    return inputs, P, tid

## Per-example forward pass


In [3]:
@torch.inference_mode()
def run_example(idx):
    inputs,P,tid=make_input(idx)
    pos=P-1

    torch.cuda.reset_peak_memory_stats()
    t0=time.time()

    on=capture_point(inputs,pos,False)
    ai.free_cuda()

    off=capture_point(inputs,pos,True)
    del inputs
    ai.free_cuda()

    rows=[]

    for L in LAYERS:
        d=(on[L]-off[L]).to(DEVICE)

        logits=ai.readout_logits(
            d,bundle,lens=lens,layer=L
        ).float()

        lp=torch.log_softmax(logits,dim=-1)

        rows.append({
            "example":idx,
            "layer":int(L),
            "target":ANS[idx]["target"],
            "target_id":tid,
            "prompt_tokens":P,
            "support_distance":int(META[idx]["support_distance_from_end"]),
            "support_pos":int(META[idx]["context_tokens"])
                          -int(META[idx]["support_distance_from_end"]),
            "placement":"evicted",
            "readout":"jlens",
            "lens_validated":LENS_VALIDATED,
            "rank_c1_residual":int(ai.token_rank(logits,tid)),
            "p_mem_c1_residual":float(torch.exp(lp[tid]).item()),
            "target_logprob":float(lp[tid].item()),
            "d_res_norm":float(d.norm().item()),
            "candidate_logprobs":{
                str(k):float(v)
                for k,v in zip(CAND_IDS,lp[CAND_T].tolist())
            }
        })

        del d,logits,lp

    sec=time.time()-t0
    peak=torch.cuda.max_memory_allocated()/1024**3

    del on,off
    ai.free_cuda()

    return rows,sec,peak



## C1/C2 statistics


In [4]:
def c2_folds(layer_rows):
    by={int(r["example"]):r for r in layer_rows}
    folds,dropped=[],0

    for a,i in enumerate(IDS):
        for j in IDS[a+1:]:
            ti,tj=ANS[i]["id"],ANS[j]["id"]

            if ti==tj:
                dropped+=1
                continue

            ri=by[i]["candidate_logprobs"]
            rj=by[j]["candidate_logprobs"]

            z=(
                (ri[str(ti)]-ri[str(tj)])
                -(rj[str(ti)]-rj[str(tj)])
            )

            folds.append(math.exp(max(min(z,700),-700)))

    return folds,dropped

def summarize(rows):
    out={}

    for L in LAYERS:
        lr=[r for r in rows if r["layer"]==L]
        ranks=[r["rank_c1_residual"] for r in lr]

        c1lo,c1hi=rc.boot_ci(ranks,st.median)

        folds,dropped=c2_folds(lr)
        g=rc.geo_mean(folds)
        flo,fhi=rc.boot_ci(folds,rc.geo_mean)

        effect=math.sqrt(g)
        elo,ehi=math.sqrt(flo),math.sqrt(fhi)

        out[str(L)]={
            "C1":{
                "n":len(ranks),
                "median_rank":float(st.median(ranks)),
                "ci95":[float(c1lo),float(c1hi)],
                "chance_rank":CHANCE,
                "below_chance":bool(c1hi<CHANCE),
                "median_target_logprob":float(
                    st.median(r["target_logprob"] for r in lr)
                )
            },
            "C2":{
                "n_pairs":len(folds),
                "dropped_same_target":dropped,
                "geometric_mean_fold":float(g),
                "effect_per_example":float(effect),
                "effect_ci95":[float(elo),float(ehi)],
                "permutation_p":float(rc.permutation_p(folds)),
                "pre_registered_bar":C2_BAR,
                "clears_bar":bool(elo>C2_BAR)
            }
        }

    return out



## Checkpoint / save state


In [5]:
def state(status, rows, done, run_log, summary=None):
    data = {
        "method_version": METHOD,
        "status": status,
        "dataset": "RMT-team/babilong",
        "config": "64k",
        "task": "qa1",
        "expected_ids": IDS,
        "completed_ids": sorted(done),
        "run_config": CFG,
        "layers": LAYERS,
        "screening_window": SCREEN_WINDOW,
        "run_window": WINDOW,
        "num_attn_sinks": SINKS,
        "lens_path": str(LENS.relative_to(REPO)),
        "lens_validated": LENS_VALIDATED,
        "rows": rows,
        "run_log": run_log,
    }

    if summary is not None:
        data["summary"] = summary

    savej(OUT, data)

## Resume + execution loop


In [6]:
rows, done, run_log = [], set(), []

if OUT.exists():
    old = loadj(OUT)

    if old.get("method_version") != METHOD:
        stamp = time.strftime("%Y%m%d_%H%M%S")
        backup = OUT.with_name(f"{OUT.stem}.incompatible_{stamp}.json")
        OUT.rename(backup)
        print("Archived incompatible partial result:", backup)

    else:
        rows = old.get("rows", [])
        done = {int(x) for x in old.get("completed_ids", [])}
        run_log = old.get("run_log", [])

        bad = {}
        for i in done:
            example_rows = [r for r in rows if int(r["example"]) == i]

            if len(example_rows) != len(LAYERS):
                bad[i] = len(example_rows)
                continue

            saved_layers = {int(r["layer"]) for r in example_rows}
            if saved_layers != set(LAYERS):
                bad[i] = sorted(saved_layers)

        if bad:
            raise RuntimeError(f"corrupt resume state: {bad}")

        print(f"Resume: {len(done)}/{len(IDS)}")

print("GPU:", torch.cuda.get_device_name(0))
print("Cohort:", len(IDS))
print("Layers:", LAYERS)
print("Window:", WINDOW, "Sinks:", SINKS)

for n, idx in enumerate(IDS, 1):
    if idx in done:
        print(f"[{n:02d}/{len(IDS)}] ID {idx}: done")
        continue

    print(f"\n[{n:02d}/{len(IDS)}] ID {idx} | {ANS[idx]['target']}")

    try:
        new, sec, gb = run_example(idx)

    except torch.cuda.OutOfMemoryError as e:
        ai.free_cuda()

        run_log.append({
            "example": idx,
            "status": "oom",
            "error": repr(e),
        })

        state("interrupted_oom", rows, done, run_log)
        raise

    rows.extend(new)
    done.add(idx)

    run_log.append({
        "example": idx,
        "status": "ok",
        "seconds": sec,
        "peak_memory_gb": gb,
    })

    for r in new:
        print(
            f" L{r['layer']}: "
            f"rank={r['rank_c1_residual']} "
            f"p={r['p_mem_c1_residual']:.2e}"
        )

    print(f" {sec:.1f}s | peak {gb:.1f}GB")
    state("running", rows, done, run_log)

Resume: 32/32
GPU: NVIDIA H100 80GB HBM3 MIG 1g.20gb
Cohort: 32
Layers: [9, 18, 27]
Window: 8064 Sinks: 128
[01/32] ID 0: done
[02/32] ID 4: done
[03/32] ID 6: done
[04/32] ID 8: done
[05/32] ID 11: done
[06/32] ID 17: done
[07/32] ID 21: done
[08/32] ID 24: done
[09/32] ID 29: done
[10/32] ID 30: done
[11/32] ID 31: done
[12/32] ID 32: done
[13/32] ID 37: done
[14/32] ID 40: done
[15/32] ID 41: done
[16/32] ID 49: done
[17/32] ID 50: done
[18/32] ID 60: done
[19/32] ID 66: done
[20/32] ID 67: done
[21/32] ID 68: done
[22/32] ID 72: done
[23/32] ID 73: done
[24/32] ID 74: done
[25/32] ID 77: done
[26/32] ID 83: done
[27/32] ID 84: done
[28/32] ID 86: done
[29/32] ID 88: done
[30/32] ID 89: done
[31/32] ID 90: done
[32/32] ID 99: done


## Final summary


In [7]:
summary = summarize(rows)
state("completed", rows, done, run_log, summary)

print("\n===== RESULT =====")

for L, s in summary.items():
    c1 = s["C1"]
    c2 = s["C2"]

    print(
        f"L{L}: "
        f"C1 {c1['median_rank']:.0f} "
        f"[{c1['ci95'][0]:.0f}, {c1['ci95'][1]:.0f}] | "
        f"C2 {c2['effect_per_example']:.2f}x "
        f"{c2['effect_ci95']} "
        f"p={c2['permutation_p']:.4g}"
    )
print("Saved:", OUT)


===== RESULT =====
L9: C1 117455 [90070, 130578] | C2 0.82x [0.7505717646690557, 0.9050805071155567] p=0
L18: C1 121582 [97081, 136081] | C2 1.42x [1.128461651640487, 1.7762730988914797] p=0.0029
L27: C1 138655 [125260, 146559] | C2 0.96x [0.8080716238584195, 1.1424358908421948] p=0.6576
Saved: /home/jupyter-dphs-29af/Interpretability-study-of-Artificial-Hippocampus-Networks/results/babilong/07c_babilong_c1c2_full.json
